# Egg Sex Classification — YOLOv5 5-Fold Cross-Validation

Fine-tunes a pretrained YOLOv5s-cls model for binary sex classification of LSCI egg images. Data must be pre-processed by `1_data_preprocessing.ipynb` before running this notebook.

**Reference:** [Roboflow YOLOv5 Classification Notebook](https://github.com/roboflow/notebooks/blob/main/notebooks/train-yolov5-classification-on-custom-data.ipynb)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pickle
import random
import shutil
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter
import torchvision.transforms as transforms
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import KFold

## 2. Configuration

In [ ]:
RANDOM_SEED  = 1839
K_FOLDS      = 5
IMG_SIZE     = 640    # YOLOv5 default; paper reports results at 640x640
EPOCHS       = 300    # max epochs per paper; early stopping at 15 epochs without improvement
BATCH_SIZE   = 64
LR           = 1e-5
OPTIMIZER    = 'Adam'
MODEL_WEIGHT = 'yolov5s-cls.pt'

# Path to pre-processed pickle files (output of 1_data_preprocessing.ipynb)
DATASET_PATH = '/content/drive/MyDrive/CS 163/egg_sex_zoomed/full_256_cv/data'

random.seed(RANDOM_SEED)

## 3. Dataset and Preprocessing

In [ ]:
def unsharp_mask(img, strength=0.95):
    """Sharpen a PIL image using an unsharp mask filter."""
    blurred    = img.filter(ImageFilter.GaussianBlur(radius=3.0))
    img_np     = np.array(img,     dtype=np.float32)
    blurred_np = np.array(blurred, dtype=np.float32)
    sharpened  = np.clip(img_np + strength * (img_np - blurred_np), 0, 255).astype(np.uint8)
    return Image.fromarray(sharpened)


class EggDataset(Dataset):
    """Loads all pickle files from a directory into memory."""
    def __init__(self, pickle_dir):
        self.data = []
        self.ids  = []
        for fname in sorted(os.listdir(pickle_dir)):
            if fname.endswith('.pickle') and not fname.startswith('.'):
                with open(os.path.join(pickle_dir, fname), 'rb') as f:
                    self.data.extend(pickle.load(f))
        self.ids = [dp['additional_info']['id'] for dp in self.data]

    def __len__(self):
        return len(self.data)


def cross_validation_splits(dataset, k_folds=5, seed=42):
    """K-fold split keeping all images of the same egg ID in the same fold."""
    unique_ids = list(set(dataset.ids))
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=seed)
    splits = []
    for train_idx, val_idx in kf.split(unique_ids):
        train_ids = {unique_ids[i] for i in train_idx}
        val_ids   = {unique_ids[i] for i in val_idx}
        train = [dp for dp in dataset.data if dp['additional_info']['id'] in train_ids]
        val   = [dp for dp in dataset.data if dp['additional_info']['id'] in val_ids]
        splits.append((train, val))
    return splits


dataset = EggDataset(DATASET_PATH)
splits  = cross_validation_splits(dataset, k_folds=K_FOLDS, seed=RANDOM_SEED)
print(f'Dataset: {len(dataset)} images, {len(set(dataset.ids))} unique egg IDs')

## 4. Build YOLOv5 Directory Structure

YOLOv5 expects images organised as:
```
datasets/fold{N}/
  train/male/   train/female/
  val/male/     val/female/
  test/male/    test/female/
```

In [ ]:
LABEL_MAP = {0: 'male', 1: 'female'}
BASE_DATASETS = '/content/datasets'


def save_fold_to_disk(fold_index, train_data, val_data):
    """Write processed images for one fold to the YOLOv5 directory layout."""
    base = os.path.join(BASE_DATASETS, f'fold{fold_index + 1}')

    for split_name, split_data in [('train', train_data), ('val', val_data)]:
        for label_id, label_name in LABEL_MAP.items():
            Path(os.path.join(base, split_name, label_name)).mkdir(parents=True, exist_ok=True)

        for dp in split_data:
            image   = unsharp_mask(dp['image'])
            label   = LABEL_MAP[dp['label']]
            img_id  = dp['additional_info']['id']
            out_dir = os.path.join(base, split_name, label)
            # Unique filename per image; append index to handle duplicate IDs
            out_path = os.path.join(out_dir, f'egg_{img_id}_{id(dp)}.png')
            image.save(out_path)

    print(f'  Fold {fold_index + 1}: {len(train_data)} train, {len(val_data)} val images written.')


print('Writing fold images to disk...')
for fold_idx, (train_data, val_data) in enumerate(splits):
    save_fold_to_disk(fold_idx, train_data, val_data)
print('Done.')

## 5. Install YOLOv5

In [ ]:
HOME = os.getcwd()
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -qr requirements.txt
import utils; utils.notebook_init()

In [ ]:
# Download YOLOv5 classification weights
from utils.downloads import attempt_download
attempt_download(f'weights/{MODEL_WEIGHT}')

In [ ]:
%cd /content/yolov5
!pip install -q albumentations

## 6. Train All Folds

Each fold's best checkpoint is saved to `runs/train-cls/fold{N}/weights/best.pt`.

In [ ]:
import os

for fold in range(1, K_FOLDS + 1):
    print(f'\n{"="*60}')
    print(f'Training Fold {fold} / {K_FOLDS}')
    print(f'{"="*60}')
    cmd = (
        f'python classify/train.py'
        f' --model {MODEL_WEIGHT}'
        f' --data /content/datasets/fold{fold}/'
        f' --imgsz {IMG_SIZE}'
        f' --epochs {EPOCHS}'
        f' --lr0 {LR}'
        f' --batch-size {BATCH_SIZE}'
        f' --optimizer {OPTIMIZER}'
        f' --project /content/runs/train-cls'
        f' --name fold{fold}'
    )
    os.system(cmd)

## 7. Evaluate All Folds

In [ ]:
import os

for fold in range(1, K_FOLDS + 1):
    weights = f'/content/runs/train-cls/fold{fold}/weights/best.pt'
    data    = f'/content/datasets/fold{fold}'
    print(f'\n--- Fold {fold} Validation ---')
    cmd = (
        f'python classify/val.py'
        f' --weights {weights}'
        f' --data {data}'
        f' --batch-size {BATCH_SIZE}'
        f' --imgsz {IMG_SIZE}'
        f' --split val'
    )
    os.system(cmd)